In [ ]:
# Creston Getz 7/13/2026
# This is the main file for the Grazioso Salvare Dashboard. 
# It is a Dash app that allows users to filter and view data from the Austin Animal Center database.
# Dash communicates to mongoDB using APIs/HTTP. app.py implments the backend of the app.


import dash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table, html
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

from dotenv import load_dotenv

import matplotlib.pyplot as plt
import requests
import pandas as pd
from app import server, df
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
load_dotenv()

# Connect to database via CRUD Module
db = AnimalShelter(
    os.environ["MONGO_USER"],
    os.environ["MONGO_PASS"],
    os.environ["MONGO_DB"],
    os.environ["MONGO_COLLECTION"],
)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################


#########################
# Dashboard Layout / View
#########################
app = dash.Dash(__name__, server=server)

#Grazisos Salvare Logo
#image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
#encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    #Style and Create the Header with the logo on the left
    html.Div(
    #    html.Div(id='hidden-div', style={'display':'none'}),
    style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'},
    children=[
        #html.Img(
            #src='data:image/png;base64,{}'.format(encoded_image.decode()), #adds and decodes logo
            #style={'width': '100px', 'height': 'auto', 'marginRight': '20px'}
        #),
        html.Div(
            style={'textAlign': 'center'},
            children=[
            html.H1('Grazioso Salvare Dashboard', style={'margin': '0'}),
            html.H2('Created by: Creston Getz', style={'margin': '0'})
            ]
        )
    ]),
    html.Hr(),
    #Section for the interactive filtering options
    html.Div([
        dcc.Dropdown(['None', 'Water Rescue', 'Mountain or Wilderness Rescue', 
                      'Disaster Rescue or Individual Tracking'], 'Filter', 
                     id='filter-type', searchable=False, placeholder="Select a Filter",),
        html.P("Select a button on the left to update the geolocation map:", 
           style={'fontWeight': 'bold', 'color': '#000000', 'marginBottom': '5px'})
        #html.Div(id='dd-output-container'),

    ]),
    html.Hr(),
    #Create the Datatable
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
        #features for interactive data table to make it user-friendly for client
        row_selectable = "single",
        selected_rows=[0],
        style_header={'fontWeight': 'bold',
                      'text-transform': 'uppercase'}, #make header clearer
        
        style_table={'overflowX': 'auto'}, #add scroll bar
        style_cell={'textAlign': 'left', 'padding_right': '20px'}, #left algin text and add padding
        
        #enable pagination
        page_action='native',
        page_current=0,
        page_size=10,
        
        #add sorting and filtering to columns
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        filter_options={"placeholder_text": "Search column..."},

    ),
    
    html.Br(),
    html.Hr(),
    #This sets up the dashboard so pie chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex', 'marginBottom':'50px'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################

#updates the datatable and chart based on the filter the user selects
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
    if filter_type == 'None':
        #output normal data and reset filters
        df = pd.DataFrame.from_records(db.read({}))
        
    elif filter_type == 'Water Rescue':
        df = pd.DataFrame.from_records(db.read({
        "animal_type": "Dog",
        "breed": {"$in":["Labrador Retriever Mix","Chesapeake Bay Retriever","Newfoundland"]},
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }))
        
    elif filter_type == 'Mountain or Wilderness Rescue':
        df = pd.DataFrame.from_records(db.read({
        "animal_type": "Dog",
        "breed": {"$in":["German Shepherd","Alaskan Malamute","Old English Sheepdog",
                         "Siberian Husky", "Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }))
        
    elif filter_type == 'Disaster Rescue or Individual Tracking':
        df = pd.DataFrame.from_records(db.read({
        "animal_type": "Dog",
        "breed": {"$in":["Doberman Pinscher","German Shepherd",
                         "Golden Retriever","Bloodhound","Rottweiler"]},
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }))
        
    else:
        df = pd.DataFrame.from_records(db.read({})) #output no filter if error
        
    df.drop(columns=['_id'], inplace=True, errors='ignore')
    return df.to_dict('records')

# Display the breeds of animal based on quantity represented in
# the data table. This method makes a pie chart based on whatever filter the user selects
# to show the distribution of dog breeds only
@app.callback(
     Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    #return nothing is no data
    if viewData is None:
        return []
    
    #Creates a dataframe of only dogs and will return a p tag if no dog data is found
    dff = pd.DataFrame.from_dict(viewData)
    dff = dff[dff['animal_type'] == 'Dog']
    if dff.empty:
        return [html.P("No dog data available for this filter.")]
    
    #Gets the top 10 breed value counts for dogs
    top_breeds = dff['breed'].value_counts().nlargest(10).reset_index()
    top_breeds.columns = ['breed', 'count']
    
    #returns figure
    return [
       dcc.Graph(            
           figure = px.pie(top_breeds, values='count', names='breed', title='Top 10 Dog Breed Distribution')
       )
    ]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            
            # Marker will display the GPS coords upon hover, along with other info about
            # the animal in a popup menu upon clicking
            dl.Marker(position=[dff.loc[row,'location_lat'],dff.loc[row,'location_long']], children=[
                dl.Tooltip(f"Lat: {dff.loc[row,'location_lat']:.4} Long: {dff.loc[row,'location_long']:.4}"),
                #dl.Popup([
                #html.H1("Animal Info"),
                #html.P(f"Name: {dff.iloc[row,9]}"),
                #html.P(f"Type: {dff.iloc[row,3]}"),
                #html.P(f"Breed: {dff.iloc[row,4]}"),
               #html.P(f"ID: {dff.iloc[row,2]}"),
               # ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(debug=True, port=8050) 

 http://127.0.0.1:8050